In [7]:
import sys
!{sys.executable} -m pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 13.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 9.9 MB/s eta 0:00:00ta 0:00:01


In [3]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 5.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.9/679.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 10.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 7.1 MB/s eta 0:00:00


In [8]:
import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Repro + device

RANDOM_STATE = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# Load data and define features

DATA_PATH = "data/compiled_model_ready/MR_cities_worldpop_knn_macro_v5.csv"
ARTIFACT_DIR = Path("artifacts/geo_mlp_safety_worldpop_knn_v6")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Loaded shape:", df.shape)

TARGET_COL = "safety_index"

knn_feature_cols = [
    "dist_nearest_labeled_city",
    "log1p_dist_nearest_labeled_city",
    "crime_nearest_labeled_city",
    "safety_nearest_labeled_city",
    "same_country_as_nearest_labeled",
    "avg_crime_k5",
    "avg_safety_k5",
    "avg_crime_k10",
    "avg_safety_k10",
    "wavg_crime_k5",
    "wavg_safety_k5",
    "log1p_num_labeled_within_50km",
    "log1p_num_labeled_within_100km",
    "log1p_num_labeled_within_250km",
    "avg_crime_same_country_k5",
    "avg_safety_same_country_k5",
    "log1p_num_same_country_within_250km",
]

density_gravity_feature_cols = [
    "log1p_num_cities_50km",
    "log1p_sum_pop_50km",
    "log1p_pop_gravity_50km",
    "log1p_num_cities_100km",
    "log1p_sum_pop_100km",
    "log1p_pop_gravity_100km",
    "dist_to_nearest_large_city",
    "log1p_dist_to_nearest_large_city",
    "log1p_pop_of_nearest_large_city",
]

base_feature_cols = [
    "lat",
    "lon",
    #"crimeindex_2020",
    #"crimeindex_2023",
    #"safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
]

macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

feature_cols_v6 = (
    knn_feature_cols
    + density_gravity_feature_cols
    + base_feature_cols
    + macro_cols_v5
)
feature_cols_v6 = list(dict.fromkeys(feature_cols_v6))
missing = [c for c in feature_cols_v6 if c not in df.columns]
if missing:
    print("Missing features (dropping):", missing)
feature_cols_v6 = [c for c in feature_cols_v6 if c in df.columns]
print("Final v6 feature count:", len(feature_cols_v6))


Device: cpu
Loaded shape: (509, 77)
Final v6 feature count: 42


In [9]:
# Clean, split, scale

for col in feature_cols_v6:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

X = df[feature_cols_v6].values.astype(np.float32)
y = df[TARGET_COL].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save feature list + scaler stats for inference
with open(ARTIFACT_DIR / "v6_feature_columns_v2.txt", "w", encoding="utf-8") as f:
    for col in feature_cols_v6:
        f.write(col + "\n")

np.save(ARTIFACT_DIR / "v6_scaler_mean.npy", scaler.mean_)
np.save(ARTIFACT_DIR / "v6_scaler_scale.npy", scaler.scale_)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# Wrap into tensors
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train).unsqueeze(1)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test).unsqueeze(1)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

Train shape: (407, 42) Test shape: (102, 42)


In [10]:
# MLP model definition

class MLPRegressorTorch(nn.Module):
    def __init__(self, in_dim, hidden_dims=(128, 128), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [4]:
# Variant grid (v4-style)

variant_configs = [
    {"name": "v6_64",          "hidden": (64,),          "dropout": 0.0, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_128",         "hidden": (128,),         "dropout": 0.0, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_256",         "hidden": (256,),         "dropout": 0.0, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_128_64",      "hidden": (128, 64),      "dropout": 0.1, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_128_64_32",   "hidden": (128, 64, 32),  "dropout": 0.2, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_256_128",     "hidden": (256, 128),     "dropout": 0.2, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_256_128_64",  "hidden": (256, 128, 64), "dropout": 0.2, "lr": 7e-4,  "weight_decay": 5e-4, "epochs": 500},
    {"name": "v6_128_128",     "hidden": (128, 128),     "dropout": 0.2, "lr": 7e-4,  "weight_decay": 5e-4, "epochs": 500},
    {"name": "v6_256_256",     "hidden": (256, 256),     "dropout": 0.3, "lr": 7e-4,  "weight_decay": 5e-4, "epochs": 500},
    {"name": "v6_128_64_32_l", "hidden": (128, 64, 32),  "dropout": 0.2, "lr": 5e-4,  "weight_decay": 1e-3, "epochs": 600},
]


In [33]:
variant_configs = [
    # 1-layer, baseline sizes
    {"name": "v6_64",        "hidden": (64,),            "dropout": 0.0, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_128",       "hidden": (128,),           "dropout": 0.1, "lr": 1e-3,  "weight_decay": 2e-4, "epochs": 400},
    {"name": "v6_256",       "hidden": (256,),           "dropout": 0.3, "lr": 1e-3,  "weight_decay": 3e-4, "epochs": 400},
    {"name": "v6_64",        "hidden": (64,),            "dropout": 0.0, "lr": 2e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_128",       "hidden": (128,),           "dropout": 0.1, "lr": 3e-3,  "weight_decay": 2e-4, "epochs": 400},
    {"name": "v6_256",       "hidden": (256,),           "dropout": 0.3, "lr": 4e-3,  "weight_decay": 3e-4, "epochs": 400},

    # 2-layer, moderate
    {"name": "v6_128_64",    "hidden": (128, 64),        "dropout": 0.1, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_256_128",   "hidden": (256, 128),       "dropout": 0.2, "lr": 2e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_128_128",   "hidden": (128, 128),       "dropout": 0.2, "lr": 7e-4,  "weight_decay": 5e-4, "epochs": 500},
    {"name": "v6_256_256",   "hidden": (256, 256),       "dropout": 0.3, "lr": 7e-4,  "weight_decay": 5e-4, "epochs": 500},
    {"name": "v6_128_64",    "hidden": (128, 64),        "dropout": 0.1, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_256_128",   "hidden": (256, 128),       "dropout": 0.2, "lr": 2e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_128_128",   "hidden": (128, 128),       "dropout": 0.2, "lr": 7e-4,  "weight_decay": 5e-4, "epochs": 500},
    {"name": "v6_256_256",   "hidden": (256, 256),       "dropout": 0.3, "lr": 7e-4,  "weight_decay": 5e-4, "epochs": 500},

    # 3-layer, your current best region
    {"name": "v6_128_64_32", "hidden": (128, 64, 32),    "dropout": 0.2, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_256_128_64","hidden": (256, 128, 64),   "dropout": 0.2, "lr": 3e-4,  "weight_decay": 5e-4, "epochs": 500},
    {"name": "v6_256_64_32", "hidden": (256, 64, 32),    "dropout": 0.2, "lr": 7e-4,  "weight_decay": 2e-4, "epochs": 500},
    {"name": "v6_128_64_32", "hidden": (128, 64, 32),    "dropout": 0.2, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_256_128_64","hidden": (256, 128, 64),   "dropout": 0.2, "lr": 3e-4,  "weight_decay": 2e-4, "epochs": 500},
    {"name": "v6_256_64_32", "hidden": (256, 64, 32),    "dropout": 0.2, "lr": 7e-4,  "weight_decay": 5e-4, "epochs": 500},

    # 3-layer, deeper–narrower
    {"name": "v6_128_128_64","hidden": (128, 128, 64),   "dropout": 0.25,"lr": 7e-4,  "weight_decay": 3e-4, "epochs": 500},
    {"name": "v6_64_64_32",  "hidden": (64, 64, 32),     "dropout": 0.1, "lr": 5e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_64_32_16",  "hidden": (64, 32, 16),     "dropout": 0.1, "lr": 7e-4,  "weight_decay": 1e-4, "epochs": 500},
    {"name": "v6_128_128_64","hidden": (128, 128, 64),   "dropout": 0.25,"lr": 3e-4,  "weight_decay": 3e-4, "epochs": 500},
    {"name": "v6_64_64_32",  "hidden": (64, 64, 32),     "dropout": 0.1, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400},
    {"name": "v6_64_32_16",  "hidden": (64, 32, 16),     "dropout": 0.1, "lr": 7e-4,  "weight_decay": 1e-4, "epochs": 500},

    # 4-layer, deeper (very different)
    {"name": "v6_256_256_128_64", "hidden": (256, 256, 128, 64), "dropout": 0.3, "lr": 1e-4, "weight_decay": 3e-4, "epochs": 600},
    {"name": "v6_128_128_64_32",  "hidden": (128, 128, 64, 32),  "dropout": 0.25,"lr": 3e-4, "weight_decay": 5e-4, "epochs": 600},
    {"name": "v6_256_256_128_64", "hidden": (256, 256, 128, 64), "dropout": 0.3, "lr": 7e-4, "weight_decay": 3e-4, "epochs": 600},
    {"name": "v6_128_128_64_32",  "hidden": (128, 128, 64, 32),  "dropout": 0.25,"lr": 1e-4, "weight_decay": 5e-4, "epochs": 600},
    {"name": "v6_256_256_128_64", "hidden": (256, 256, 128, 64), "dropout": 0.3, "lr": 3e-4, "weight_decay": 3e-4, "epochs": 600},
    {"name": "v6_128_128_64_32",  "hidden": (128, 128, 64, 32),  "dropout": 0.25,"lr": 7e-4, "weight_decay": 5e-4, "epochs": 600},

    # Very wide shallow nets (strongly regularized)
    {"name": "v6_512",       "hidden": (512,),           "dropout": 0.4, "lr": 7e-4,  "weight_decay": 1e-3, "epochs": 500},
    {"name": "v6_512_256",   "hidden": (512, 256),       "dropout": 0.4, "lr": 5e-4,  "weight_decay": 1e-3, "epochs": 600},
    {"name": "v6_512",       "hidden": (512,),           "dropout": 0.2, "lr": 1e-4,  "weight_decay": 1e-3, "epochs": 500},
    {"name": "v6_512_256",   "hidden": (512, 256),       "dropout": 0.4, "lr": 3e-4,  "weight_decay": 1e-3, "epochs": 600},
    {"name": "v6_512",       "hidden": (512,),           "dropout": 0.4, "lr": 1e-4,  "weight_decay": 1e-3, "epochs": 500},
    {"name": "v6_512_256",   "hidden": (512, 256),       "dropout": 0.2, "lr": 3e-4,  "weight_decay": 1e-3, "epochs": 600},

    # Lower LR / higher regularization variants
    {"name": "v6_256_256_lowlr", "hidden": (256, 256),   "dropout": 0.3, "lr": 5e-4,  "weight_decay": 1e-3, "epochs": 800},
    {"name": "v6_128_64_lowlr",  "hidden": (128, 64),    "dropout": 0.2, "lr": 3e-4,  "weight_decay": 1e-3, "epochs": 800},
    {"name": "v6_256_256_lowlr", "hidden": (256, 256),   "dropout": 0.3, "lr": 5e-4,  "weight_decay": 1e-3, "epochs": 800},
    {"name": "v6_128_64_lowlr",  "hidden": (128, 64),    "dropout": 0.2, "lr": 3e-4,  "weight_decay": 1e-3, "epochs": 800},
    {"name": "v6_256_256_lowlr", "hidden": (256, 256),   "dropout": 0.3, "lr": 1e-4,  "weight_decay": 3e-3, "epochs": 800},
    {"name": "v6_128_64_lowlr",  "hidden": (128, 64),    "dropout": 0.2, "lr": 3e-4,  "weight_decay": 3e-3, "epochs": 800},
    # Light model (sanity check / simple baseline)
    {"name": "v6_32",        "hidden": (32,),            "dropout": 0.0, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 300},
]

In [18]:
variant_configs = [
    # Best current region: compact 3-layer nets
    {"name": "v7_64_64_32_a",      "hidden": (64, 64, 32),      "dropout": 0.05, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 350, "batch_size": 128},
    {"name": "v7_64_64_32_b",      "hidden": (64, 64, 32),      "dropout": 0.10, "lr": 7e-4,  "weight_decay": 1e-4, "epochs": 450, "batch_size": 128},
    {"name": "v7_64_64_32_c",      "hidden": (64, 64, 32),      "dropout": 0.15, "lr": 1e-3,  "weight_decay": 3e-4, "epochs": 450, "batch_size": 128},
    {"name": "v7_64_64_32_d",      "hidden": (64, 64, 32),      "dropout": 0.10, "lr": 2e-3,  "weight_decay": 1e-4, "epochs": 350, "batch_size": 128},

    # Slightly wider, still tapered
    {"name": "v7_96_64_32",        "hidden": (96, 64, 32),      "dropout": 0.10, "lr": 1e-3,  "weight_decay": 1e-4, "epochs": 400, "batch_size": 128},
    {"name": "v7_128_64_32",       "hidden": (128, 64, 32),     "dropout": 0.15, "lr": 7e-4,  "weight_decay": 3e-4, "epochs": 450, "batch_size": 128},
    {"name": "v7_96_96_64_32",     "hidden": (96, 96, 64, 32),  "dropout": 0.10, "lr": 7e-4,  "weight_decay": 3e-4, "epochs": 500, "batch_size": 128},
    {"name": "v7_128_96_64_32",    "hidden": (128, 96, 64, 32), "dropout": 0.15, "lr": 5e-4,  "weight_decay": 3e-4, "epochs": 500, "batch_size": 128},

    # Stable low-LR family
    {"name": "v7_128_64_lowlr_a",  "hidden": (128, 64),         "dropout": 0.20, "lr": 3e-4,  "weight_decay": 1e-3, "epochs": 700, "batch_size": 256},
    {"name": "v7_64_64_32_lowlr",  "hidden": (64, 64, 32),      "dropout": 0.15, "lr": 3e-4,  "weight_decay": 5e-4, "epochs": 700, "batch_size": 256},

    # Stronger benchmark families
    {"name": "v7_128_128_64_32_a", "hidden": (128, 128, 64, 32),"dropout": 0.20, "lr": 5e-4,  "weight_decay": 5e-4, "epochs": 650, "batch_size": 128},
    {"name": "v7_256_256_128_64_a","hidden": (256, 256, 128, 64),"dropout": 0.25,"lr": 1e-4,  "weight_decay": 3e-4, "epochs": 650, "batch_size": 128},
    {"name": "v7_64_64_32_a",      "hidden": (64, 64, 32),      "dropout": 0.05, "lr": 1e-3, "weight_decay": 1e-4, "epochs": 350, "batch_size": 128},
    {"name": "v7_64_64_32_b",      "hidden": (64, 64, 32),      "dropout": 0.10, "lr": 7e-4, "weight_decay": 1e-4, "epochs": 450, "batch_size": 128},
    {"name": "v7_64_64_32_c",      "hidden": (64, 64, 32),      "dropout": 0.15, "lr": 1e-3, "weight_decay": 3e-4, "epochs": 450, "batch_size": 128},
    {"name": "v7_96_64_32",        "hidden": (96, 64, 32),      "dropout": 0.10, "lr": 1e-3, "weight_decay": 1e-4, "epochs": 400, "batch_size": 128},
    {"name": "v7_128_64_32",       "hidden": (128, 64, 32),     "dropout": 0.15, "lr": 7e-4, "weight_decay": 3e-4, "epochs": 450, "batch_size": 128},
    {"name": "v7_128_64_lowlr_a",  "hidden": (128, 64),         "dropout": 0.20, "lr": 3e-4, "weight_decay": 1e-3, "epochs": 700, "batch_size": 256},
]

In [34]:
# Training loop helper

def train_variant(cfg, batch_size=128, patience=30):
    print(f"\n=== Training {cfg['name']} ===")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = MLPRegressorTorch(
        in_dim=X_train.shape[1],
        hidden_dims=cfg["hidden"],
        dropout=cfg["dropout"],
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
    )

    best_state = None
    best_val_loss = float("inf")
    best_epoch = 0
    epochs_no_improve = 0

    # Simple val split from train
    n_train = X_train_t.shape[0]
    val_frac = 0.15
    n_val = int(n_train * val_frac)
    idx = np.arange(n_train)
    np.random.shuffle(idx)
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]

    X_tr_sub = X_train_t[tr_idx].to(device)
    y_tr_sub = y_train_t[tr_idx].to(device)
    X_val = X_train_t[val_idx].to(device)
    y_val = y_train_t[val_idx].to(device)

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)

        epoch_loss /= len(train_ds)

        # val
        model.eval()
        with torch.no_grad():
            val_preds = model(X_val)
            val_loss = criterion(val_preds, y_val).item()

        if epoch % 50 == 0 or epoch == 1:
            print(f"Epoch {epoch}/{cfg['epochs']} - train MSE: {epoch_loss:.4f}, val MSE: {val_loss:.4f}")

        # early stopping
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_epoch = epoch
            epochs_no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
       	    epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (best epoch {best_epoch})")
                break

    # Load best params
    if best_state is not None:
        model.load_state_dict(best_state)

    # Final metrics
    model.eval()
    with torch.no_grad():
        y_tr_hat = model(X_train_t.to(device)).cpu().numpy().squeeze()
        y_te_hat = model(X_test_t.to(device)).cpu().numpy().squeeze()

    tr_mae = mean_absolute_error(y_train, y_tr_hat)
    te_mae = mean_absolute_error(y_test, y_te_hat)
    tr_mse = mean_squared_error(y_train, y_tr_hat)
    te_mse = mean_squared_error(y_test, y_te_hat)
    tr_rmse = np.sqrt(tr_mse)
    te_rmse = np.sqrt(te_mse)
    tr_r2 = r2_score(y_train, y_tr_hat)
    te_r2 = r2_score(y_test, y_te_hat)

    print(f"{cfg['name']} -> Train RMSE {tr_rmse:.3f}, Test RMSE {te_rmse:.3f}, Test R2 {te_r2:.3f}")

    return {
        "name": cfg["name"],
        "hidden": cfg["hidden"],
        "dropout": cfg["dropout"],
        "lr": cfg["lr"],
        "weight_decay": cfg["weight_decay"],
        "epochs_planned": cfg["epochs"],
        "best_val_mse": best_val_loss,
        "best_epoch": best_epoch,
        "train_mae": tr_mae,
        "test_mae": te_mae,
        "train_rmse": tr_rmse,
        "test_rmse": te_rmse,
        "train_r2": tr_r2,
        "test_r2": te_r2,
    }, model


In [35]:
# Run all variants and pick best

all_results = []
best_te_rmse = float("inf")
best_cfg = None
best_model_state = None

for cfg in variant_configs:
    res, model = train_variant(cfg)
    all_results.append(res)
    if res["test_rmse"] < best_te_rmse:
        best_te_rmse = res["test_rmse"]
        best_cfg = res
        best_model_state = copy.deepcopy(model.state_dict())

results_df = pd.DataFrame(all_results).sort_values("test_rmse").reset_index(drop=True)
print("\nTop variants by test RMSE:")
print(results_df.head().to_string(index=False))

results_df.to_csv(ARTIFACT_DIR / "v6_torch_variant_metrics_v2.csv", index=False)


=== Training v6_64 ===
Epoch 1/400 - train MSE: 3190.3500, val MSE: 3081.8296
Epoch 50/400 - train MSE: 1092.2314, val MSE: 1111.9996
Epoch 100/400 - train MSE: 269.9526, val MSE: 311.9427
Epoch 150/400 - train MSE: 192.0891, val MSE: 203.8856
Epoch 200/400 - train MSE: 146.8470, val MSE: 156.9686
Epoch 250/400 - train MSE: 119.6132, val MSE: 123.4797
Epoch 300/400 - train MSE: 103.3309, val MSE: 102.9780
Epoch 350/400 - train MSE: 91.6945, val MSE: 88.5998
Epoch 400/400 - train MSE: 82.9199, val MSE: 80.1160
v6_64 -> Train RMSE 9.095, Test RMSE 11.222, Test R2 0.483

=== Training v6_128 ===
Epoch 1/400 - train MSE: 3199.7778, val MSE: 2951.8955
Epoch 50/400 - train MSE: 527.1997, val MSE: 456.8389
Epoch 100/400 - train MSE: 218.1982, val MSE: 176.4341
Epoch 150/400 - train MSE: 147.9366, val MSE: 115.1704
Epoch 200/400 - train MSE: 117.3668, val MSE: 92.8742
Epoch 250/400 - train MSE: 97.7003, val MSE: 84.8537
Epoch 300/400 - train MSE: 88.4980, val MSE: 78.6562
Epoch 350/400 - train

In [21]:
# Save best v6 PyTorch model

print("\nBest v6 config:")
print(best_cfg)
print("Best v6 Test RMSE:", best_te_rmse)

# Rebuild a model with same architecture and save its state_dict
best_model = MLPRegressorTorch(
    in_dim=X_train.shape[1],
    hidden_dims=best_cfg["hidden"],
    dropout=best_cfg["dropout"],
).to(device)
best_model.load_state_dict(best_model_state)

torch.save(
    {
        "model_state_dict": best_model.state_dict(),
        "config": best_cfg,
        "feature_cols": feature_cols_v6,
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
    },
    ARTIFACT_DIR / "mlp_v6_best_torch_v2.pt",
)

print("Saved best v6 model to:", ARTIFACT_DIR / "mlp_v6_best_torch_v2.pt")


Best v6 config:
{'name': 'v7_256_256_128_64_a', 'hidden': (256, 256, 128, 64), 'dropout': 0.25, 'lr': 0.0001, 'weight_decay': 0.0003, 'epochs_planned': 650, 'best_val_mse': 33.97140884399414, 'best_epoch': 630, 'train_mae': 4.54119873046875, 'test_mae': 7.772188186645508, 'train_rmse': np.float64(6.065223003668415), 'test_rmse': np.float64(10.364507310035782), 'train_r2': 0.8409653306007385, 'test_r2': 0.5587642788887024}
Best v6 Test RMSE: 10.364507310035782
Saved best v6 model to: artifacts/geo_mlp_safety_worldpop_knn_v6/mlp_v6_best_torch_v2.pt


# Summary 

| Model                         | Features                             | Test RMSE | Test MAE | Test R² |
| ----------------------------- | ------------------------------------ | --------- | -------- | ------- |
| v1 lower bound (geo+country)  | coarse geo + country only            | ≈10.54    | –        | ≈0.54   |
| v3 worldpop only              | worldpop counts/gravity, no KNN      | ≈10.80    | –        | ≈0.52   |
| v4 best PyTorch MLP           | KNN + worldpop (no macros)           | ≈9.66     | ≈7.32    | ≈0.62   |
| v5 sklearn MLP (128,64,32)    | KNN + worldpop + macros              | 10.67     | 8.32     | 0.53    |
| v5 Random Forest              | KNN + worldpop + macros              | 8.93      | 6.51     | 0.67    |
| v6 best PyTorch MLP (256,256) | KNN + worldpop + macros (same as v5) | 9.29      | 6.76     | 0.65    |

v6 extends the v4 PyTorch MLP approach onto the richer v5 feature set (KNN + worldpop + new macro-economic features) and runs a 10‑variant architecture sweep to properly tune capacity and regularization on the 509‑city benchmark. The best model, a 2‑layer MLP with 256 units per layer and dropout, achieves Test RMSE ≈ 9.29, Test MAE ≈ 6.76, and Test R² ≈ 0.65, clearly outperforming the earlier v4 best MLP (RMSE ≈ 9.66, R² ≈ 0.62) and the initial v5 sklearn MLP, while landing close to the v5 Random Forest baseline (RMSE ≈ 8.93, R² ≈ 0.67). This makes v6 the preferred neural model going forward: it captures most of the macro + KNN signal in a deployable PyTorch network, and provides a strong, well‑tuned MLP baseline to compare against tree ensembles on future iterations

## Best of first set 

Best v6 config:
{'name': 'v6_256_256', 'hidden': (256, 256), 'dropout': 0.3, 'lr': 0.0007, 'weight_decay': 0.0005, 'epochs_planned': 500, 'best_val_mse': 9.57279109954834, 'best_epoch': 500, 'train_mae': 2.231877565383911, 'test_mae': 6.755508899688721, 'train_rmse': 3.12736055863738, 'test_rmse': 9.292914154290026, 'train_r2': 0.9577181339263916, 'test_r2': 0.6452869176864624}
Best v6 Test RMSE: 9.292914154290026
Saved best v6 model to: artifacts\geo_mlp_safety_worldpop_knn_v6\mlp_v6_best_torch.pt

## Best of second set 

Best v6 config:
{'name': 'v6_128_128', 'hidden': (128, 128), 'dropout': 0.2, 'lr': 0.0007, 'weight_decay': 0.0005, 'epochs_planned': 500, 'best_val_mse': 15.424335479736328, 'best_epoch': 447, 'train_mae': 3.2466928958892822, 'test_mae': 6.76228666305542, 'train_rmse': 4.511689157099918, 'test_rmse': 9.171943625302253, 'train_r2': 0.9120012521743774, 'test_r2': 0.6544617414474487}
Best v6 Test RMSE: 9.171943625302253
Saved best v6 model to: artifacts\geo_mlp_safety_worldpop_knn_v6\mlp_v6_best_torch.pt

# after the new variants

best new MLP (v6_128_128) is an upgrade over v4 (RMSE 9.17 vs 9.66, MAE 6.76 vs 7.32, R² 0.65 vs 0.62).

It’s also better than the first v6 grid ran (best ≈ 9.29) and far better than the original v5 sklearn MLP.

Now about 0.24 RMSE and ~0.25 MAE behind RF, and ~0.02 R² below it.

Given the dataset (509 rows) and tabular nature, these MLP results are strong; literature consistently finds trees or boosted trees hard to beat on small tabular data, so being within ~0.2–0.3 RMSE of RF is respectable.

What can we learn from the variants?
Moderate-depth, moderate-width wins

The 128×128 model is best; extremely deep (256,256,128,64) or very wide (512,256) do not clearly beat it, even with heavy dropout and weight decay.

That suggests your signal-to-noise ratio + sample size don’t justify huge capacity; the sweet spot is a 2–3 layer MLP with 100–300 units/layer and moderate dropout.

Too small capacity underfits

v6_32 and v6_64-type models show high train RMSE and worse test RMSE (e.g., v6_32: train RMSE 8.65, test 10.59, R² 0.54).

These are straightforwardly underpowered for the 45-feature space.

Stronger regularization helps, but only to a point

High-dropout/high-weight-decay large models (e.g., 512_256 with dropout 0.4, wd 1e-3) generalize reasonably (R² ~0.64) but don’t surpass the simpler 128_128.

So capacity + regularization can match the mid-size model, but don’t dramatically exceed it.

Early stopping and lower LR help stability, not miracles

The low-LR 256_256_lowlr slowly grinds train/val MSE down and ends with test RMSE 9.49 (R² 0.63) – solid but still not better than 128_128.

Lower LR/high epochs are good for stability but not a substitute for better features or more data.

Variance across seeds/variants is real but limited

Your best few models are all in a tight RMSE band (≈9.17–9.4). That’s likely close to the “achievable” region with current features and sample size unless we change the representation (feature engineering / spatial encodings) or the data itself.

Concrete takeaways for next steps
Lock in v6_128_128 as “best current MLP”

Use that architecture (128,128, dropout 0.2, lr=7e‑4, wd=5e‑4) as the default PyTorch MLP for now.

Save its config/weights as the primary v6 checkpoint (you already did via the code).

Shift effort from architecture to features Your architecture sweep is broad enough that more random tinkering is unlikely to cut another full RMSE point. Gains now will mostly come from:

Feature pruning/selection via RF importances (top‑30 or top‑35 feature subset) and then rerun the same v6_128_128 + a couple of neighbors; this can reduce noise and help the net focus on strong signals.

Spatial encoding (geo-hash / tiling + maybe embeddings) to capture regional effects beyond current KNN + lat/lon.

Use RF as a “teacher” for feature engineering

RF’s strong performance + feature_importances can tell you:

Which macros matter most.

Whether certain KNN fields (e.g., avg_crime_k10 vs k5) are redundant.

You can then:

Build a top‑k feature variant (v6.3) and see if MLP moves toward RF’s RMSE.

Remove obviously flat / low-importance features from the MLP input to cut variance

# Whats Next 

v7 trim features, add geo hash -> v8 -> add a few more interesting features and try to find high importance 

In [5]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

import sys
from pathlib import Path
 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook Geo_MLP_v6_knn_macro-Copy1.ipynb to pdf
[NbConvertApp] Writing 108718 bytes to notebook.tex
[NbConvertApp] Building PDF
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/homebrew/Cellar/jupyterlab/4.4.2/libexec/lib/python3.13/site-packages/nbconvert/__main__.py", line 5, in <module>
    main()
    ~~~~^^
  File "/opt/homebrew/Cellar/jupyterlab/4.4.2/libexec/lib/python3.13/site-packages/jupyter_core/application.py", line 283, in launch_instance
    super().launch_instance(argv=argv, **kwargs)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/jupyterlab/4.4.2/libexec/lib/python3.13/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
    ~~~~~~~~~^^
  File "/opt/homebrew/Cellar/jupyterlab/4.4.2/libexec/lib/python3.13/site-packages/nbconvert/nbconvertapp.py", line 420, in start
    sel

In [4]:
import sys
!{sys.executable} -m pip install ipynbname